In [30]:
import sys
sys.path.insert(0, "/home/nfm/ViT-Prisma/src")

from vit_prisma.sae import VisionModelSAERunnerConfig
from vit_prisma.sae import VisionSAETrainer
from vit_prisma.transforms import get_clip_val_transforms


import torchvision
import torch

from torch.utils.data import DataLoader, Subset


# Train the SAE

In [2]:

# Load an SAE
from huggingface_hub import hf_hub_download, list_repo_files
from vit_prisma.sae import SparseAutoencoder

def load_sae(repo_id, file_name, config_name):
    # Step 1: Download SAE weights and SAE config from Hugginface
    sae_path = hf_hub_download(repo_id, file_name) # Download SAE weights
    hf_hub_download(repo_id, config_name) # Download SAE config

    # Step 2: Now load the pretrained SAE weights from where you just downloaded them
    print(f"Loading SAE from {sae_path}...")
    sae = SparseAutoencoder.load_from_pretrained(sae_path) # This now automatically gets the config.json file in that folder and converts it into a VisionSAERunnerConfig object
    return sae

# Change the repo_id to the Huggingface repo of your chosen SAE. See /docs for a list of SAEs.
repo_id = "Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05" 

file_name = "weights.pt" # Usually weights.pt but may have slight naming variation. See the chosen HF repo for the exact file name
config_name = "config.json"

sae = load_sae(repo_id, file_name, config_name)

2026-04-20 12:10:48 DEBUG:urllib3.connectionpool: Starting new HTTPS connection (1): huggingface.co:443
2026-04-20 12:10:48 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/resolve/main/weights.pt HTTP/1.1" 302 0
2026-04-20 12:10:49 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/resolve/main/config.json HTTP/1.1" 307 0
2026-04-20 12:10:49 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/Prisma-Multimodal/sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/1e0dcc98c63aed3ce28f0387027cf8df6784fef1/config.json HTTP/1.1" 200 0


Loading SAE from /home/nfm/.cache/huggingface/hub/models--Prisma-Multimodal--sparse-autoencoder-clip-b-32-sae-vanilla-x64-layer-10-hook_mlp_out-l1-1e-05/snapshots/1e0dcc98c63aed3ce28f0387027cf8df6784fef1/weights.pt...


2026-04-20 12:10:49 WARNING:root: Deprecated field 'total_training_images' found in config. It will be ignored.
2026-04-20 12:10:49 WARNING:root: Deprecated field 'total_training_tokens' found in config. It will be ignored.
2026-04-20 12:10:49 WARNING:root: Deprecated field 'd_sae' found in config. It will be ignored.
2026-04-20 12:10:49 INFO:root: n_tokens_per_buffer (millions): 0.032
2026-04-20 12:10:49 INFO:root: Lower bound: n_contexts_per_buffer (millions): 0.00064
2026-04-20 12:10:49 INFO:root: Total training steps: 158691
2026-04-20 12:10:49 INFO:root: Total training images: 13000000
2026-04-20 12:10:49 INFO:root: Total wandb updates: 1586
2026-04-20 12:10:49 INFO:root: Expansion factor: 64
2026-04-20 12:10:49 INFO:root: n_tokens_per_feature_sampling_window (millions): 204.8
2026-04-20 12:10:49 INFO:root: n_tokens_per_dead_feature_window (millions): 1024.0
2026-04-20 12:10:49 INFO:root: We will reset the sparsity calculation 158 times.
2026-04-20 12:10:49 INFO:root: Number token

In [4]:
from pprint import pprint
sae.cfg
pprint(sae.cfg)

VisionModelSAERunnerConfig(model_class_name='HookedViT',
                           model_name='open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K',
                           vit_model_cfg=None,
                           model_path=None,
                           hook_point_layer=10,
                           layer_subtype='hook_mlp_out',
                           hook_point_head_index=None,
                           context_size=50,
                           use_cached_activations=False,
                           use_patches_only=False,
                           cached_activations_path='activations/_network_scratch_s_sonia.joseph_datasets_kaggle_datasets/open-clip:laion_CLIP-ViT-B-32-DataComp.XL-s13B-b90K/blocks.9.hook_mlp_out',
                           image_size=224,
                           architecture='standard',
                           b_dec_init_method='geometric_median',
                           expansion_factor=64,
                           from_pretrained

In [6]:
from vit_prisma.models.model_loader import load_hooked_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = sae.cfg.model_name
# model_name = "vit_base_patch16_224"
model = load_hooked_model(model_name)
model.to(DEVICE);

2026-04-20 12:12:37 INFO:root: Model 'open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K' is supported and passes tests.
2026-04-20 12:12:37 INFO:root: model_id download_pretrained_from_hf: laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K
2026-04-20 12:12:37 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K/resolve/main/open_clip_config.json HTTP/1.1" 307 0
2026-04-20 12:12:37 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K/f0e2ffa09cbadab3db6a261ec1ec56407ce42912/open_clip_config.json HTTP/1.1" 200 0


*****Loading model 'open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K' of type 'VISION'


2026-04-20 12:12:38 INFO:root: model_id download_pretrained_from_hf: laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K
2026-04-20 12:12:38 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K/resolve/main/open_clip_pytorch_model.bin HTTP/1.1" 302 0
2026-04-20 12:12:38 INFO:root: visual projection shape: torch.Size([768, 512])
2026-04-20 12:12:38 INFO:root: Loaded pretrained model open-clip:laion/CLIP-ViT-B-32-DataComp.XL-s13B-b90K into HookedTransformer


In [7]:
# Put your ImageNet Paths here
from vit_prisma.transforms import get_clip_val_transforms

imagenet_train_path = '/home/nfm/data_prisma/imagenet_val/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'
imagenet_validation_path = '/home/nfm/data_prisma/imagenet_val/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'

data_transforms = get_clip_val_transforms()
train_dataset = torchvision.datasets.ImageFolder(imagenet_train_path, transform=data_transforms)
eval_dataset = torchvision.datasets.ImageFolder(imagenet_validation_path, transform=data_transforms)


In [31]:
import torchvision
from torchvision import transforms

# Standard ImageNet transforms for timm models
data_transforms = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

imagenet_train_path = '/home/nfm/data_prisma/imagenet_val/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'
imagenet_validation_path = '/home/nfm/data_prisma/imagenet_val/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/val'

train_dataset = torchvision.datasets.ImageFolder(imagenet_train_path, transform=data_transforms)
eval_dataset = torchvision.datasets.ImageFolder(imagenet_validation_path, transform=data_transforms)

In [33]:
MODEL_NAME = "vit_base_patch16_224" # Just define it here!

# 1. Create your full, standalone config
sae_trainer_cfg = VisionModelSAERunnerConfig( 
    model_name=MODEL_NAME, # <--- Pass the model name here
    hook_point_layer=11,
    layer_subtype='hook_resid_post',
    dataset_name="imagenet",
    feature_sampling_window=1000,
    activation_fn_str='relu',
    cls_token_only=True,
    context_size=1,
    num_workers=6,
    store_batch_size=256,   
    train_batch_size=8192,  
    checkpoint_path='/home/nfm/ViT-Prisma/demos/sae_ckpts',
    num_epochs=10,
    n_checkpoints=5
)

from vit_prisma.models.model_loader import load_hooked_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = load_hooked_model(MODEL_NAME)
model.to(DEVICE);

2026-04-20 17:00:57 INFO:root: n_tokens_per_buffer (millions): 0.00512
2026-04-20 17:00:57 INFO:root: Lower bound: n_contexts_per_buffer (millions): 0.00512
2026-04-20 17:00:57 INFO:root: Total training steps: 1586
2026-04-20 17:00:57 INFO:root: Total training images: 13000000
2026-04-20 17:00:57 INFO:root: Total wandb updates: 158
2026-04-20 17:00:57 INFO:root: Expansion factor: 16
2026-04-20 17:00:57 INFO:root: n_tokens_per_feature_sampling_window (millions): 8.192
2026-04-20 17:00:57 INFO:root: n_tokens_per_dead_feature_window (millions): 40.96
2026-04-20 17:00:57 INFO:root: We will reset the sparsity calculation 1 times.
2026-04-20 17:00:57 INFO:root: Number tokens in sparsity calculation window: 8.19e+06
2026-04-20 17:00:57 INFO:root: Gradient clipping with max_norm=1.0
2026-04-20 17:00:57 INFO:root: Using SAE initialization method: independent
2026-04-20 17:00:57 WARNING:root: Model 'vit_base_patch16_224' is not in the lists of models passing or failing tests. Unclear status. You

*****Loading model 'vit_base_patch16_224' of type 'VISION'


/home/nfm/prisma/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning:

`resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.

2026-04-20 17:00:58 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/config.json HTTP/1.1" 307 0
2026-04-20 17:00:58 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /api/resolve-cache/models/timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/063c6c38a5d8510b2e57df480445e94b231dad2c/config.json HTTP/1.1" 200 0


ln_pre not set


2026-04-20 17:01:00 INFO:timm.models._builder: Loading pretrained weights from Hugging Face hub (timm/vit_base_patch16_224.augreg2_in21k_ft_in1k)
2026-04-20 17:01:00 DEBUG:urllib3.connectionpool: https://huggingface.co:443 "HEAD /timm/vit_base_patch16_224.augreg2_in21k_ft_in1k/resolve/main/model.safetensors HTTP/1.1" 302 0
2026-04-20 17:01:00 INFO:timm.models._hub: [timm/vit_base_patch16_224.augreg2_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


Converting the weights of a timm model to a Prisma ViT


2026-04-20 17:01:01 INFO:root: Loaded pretrained model vit_base_patch16_224 into HookedTransformer


In [28]:
# sae_trainer_cfg = VisionModelSAERunnerConfig( 
#     hook_point_layer=10,
#     layer_subtype='hook_resid_post',
#     dataset_name="imagenet",
#     feature_sampling_window=1000,
#     activation_fn_str='relu',
#     cls_token_only=True,
#     num_workers=6,
#     context_size=1,
#     store_batch_size=256,   # Speeds up ViT inference
#     train_batch_size=8192,  # Speeds up SAE training
#     checkpoint_path = '/home/nfm/ViT-Prisma/demos/sae_ckpts',
#     num_epochs=10
# )

In [34]:
pprint(sae_trainer_cfg)
# !ls /home/nfm/ViT-Prisma/demos/sae_ckpts/ee158df6-tinyclip_sae_16_hyperparam_sweep_lr

VisionModelSAERunnerConfig(model_class_name='HookedViT',
                           model_name='vit_base_patch16_224',
                           vit_model_cfg=None,
                           model_path=None,
                           hook_point_layer=11,
                           layer_subtype='hook_resid_post',
                           hook_point_head_index=None,
                           context_size=1,
                           use_cached_activations=False,
                           use_patches_only=False,
                           cached_activations_path='activations/_network_scratch_s_sonia.joseph_datasets_kaggle_datasets/vit_base_patch16_224/blocks.11.hook_resid_post',
                           image_size=224,
                           architecture='standard',
                           b_dec_init_method='geometric_median',
                           expansion_factor=16,
                           from_pretrained_path=None,
                           is_transcoder=Fal

In [35]:
trainer = VisionSAETrainer(sae_trainer_cfg, model, train_dataset, eval_dataset)
sae = trainer.run()

2026-04-20 17:01:53 INFO:root: get_activation_fn received: activation_fn=relu, kwargs={}


details/current_learning_rate,▃▃▃▄▄▅▅▅▆▆▇▇██████▇▇▆▅▅▅▄▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁
details/n_training_images,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇███
details/n_training_tokens,▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
losses/ghost_grad_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/l1_loss,█▇▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/mse_loss,█▆▅▅▅▄▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/overall_loss,█▇▇▆▅▅▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
metrics/explained_variance,▁▂▃▄▅▅▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████████████
metrics/explained_variance_std,▁▁▁▁▂█▇▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
metrics/l0,█▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


2026-04-20 17:02:24 DEBUG:git.util: sys.platform='linux', git_executable='git'
2026-04-20 17:02:24 DEBUG:git.cmd: Popen(['git', 'cat-file', '--batch-check'], cwd=/home/nfm/ViT-Prisma, stdin=<valid stream>, shell=False, universal_newlines=False)


Objective value: 129608.9062:   2%|▏         | 3/200 [00:00<00:00, 263.23it/s]
Training SAE: Loss: 0.7950, MSE Loss: 0.4363, L1 Loss: 0.3587, L0: 6132.5605:   1%|          | 81920/13000000 [01:16<3:26:57, 1040.29it/s]2026-04-20 17:03:41 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 22 - Data Location: 4688 - value: b'EASTMAN KODAK COMPANY\x00'
2026-04-20 17:03:41 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 34 - Data Location: 4710 - value: b'KODAK C310 DIGITAL CAMERA\x00'
2026-04-20 17:03:41 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 46 - Data Location: 206 - value: b'\xe6\x00\x00\x00\x01\x00\x00\x00'
2026-04-20 17:03:41 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 58 - Data Location: 214 - value: b'\xe6\x00\x00\x00\x01\x00\x00\x00'
2026-04-20 17:03:41 DEBUG:PIL.TiffImagePlugin: tag: ResolutionUnit (296) - type: short (3) - value: b'\x02\x00'


Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/f103614d-tinyclip_sae_16_hyperparam_sweep_lr/n_images_2605056.pt


...

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/f103614d-tinyclip_sae_16_hyperparam_sweep_lr/n_images_7806976.pt


Training SAE: Loss: 0.0238, MSE Loss: 0.0127, L1 Loss: 0.0112, L0: 106.4668:  60%|██████    | 7856128/13000000 [2:18:09<1:29:31, 957.54it/s]2026-04-20 19:20:42 DEBUG:PIL.TiffImagePlugin: tag: ImageDescription (270) - type: string (2) Tag Location: 22 - Data Location: 134 - value: <table: 33 bytes>
2026-04-20 19:20:42 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 34 - Data Location: 167 - value: b'NIKON CORPORATION\x00'
2026-04-20 19:20:42 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 46 - Data Location: 185 - value: b'NIKON D100 \x00'
2026-04-20 19:20:42 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 58 - Data Location: 197 - value: b'\x00\x00\x00H\x00\x00\x00\x01'
2026-04-20 19:20:42 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 70 - Data Location: 205 - value: b'\x00\x00\x00H\x00\x00\x00\x01'
2026-04-20 19:20:42 DEBUG:PIL.TiffImagePlugin: tag: Reso

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/f103614d-tinyclip_sae_16_hyperparam_sweep_lr/n_images_10403840.pt


Training SAE: Loss: 0.0228, MSE Loss: 0.0120, L1 Loss: 0.0108, L0: 108.5988:  82%|████████▏ | 10665984/13000000 [3:08:20<40:39, 956.96it/s]2026-04-20 20:10:57 DEBUG:PIL.TiffImagePlugin: tag: Make (271) - type: string (2) Tag Location: 22 - Data Location: 122 - value: b'Canon\x00'
2026-04-20 20:10:57 DEBUG:PIL.TiffImagePlugin: tag: Model (272) - type: string (2) Tag Location: 34 - Data Location: 128 - value: b'Canon EOS DIGITAL REBEL XT\x00'
2026-04-20 20:10:57 DEBUG:PIL.TiffImagePlugin: tag: Orientation (274) - type: short (3) - value: b'\x01\x00'
2026-04-20 20:10:57 DEBUG:PIL.TiffImagePlugin: tag: XResolution (282) - type: rational (5) Tag Location: 58 - Data Location: 155 - value: b'H\x00\x00\x00\x01\x00\x00\x00'
2026-04-20 20:10:57 DEBUG:PIL.TiffImagePlugin: tag: YResolution (283) - type: rational (5) Tag Location: 70 - Data Location: 163 - value: b'H\x00\x00\x00\x01\x00\x00\x00'
2026-04-20 20:10:57 DEBUG:PIL.TiffImagePlugin: tag: ResolutionUnit (296) - type: short (3) - value: b'\x

Saved SAE to /home/nfm/ViT-Prisma/demos/sae_ckpts/f103614d-tinyclip_sae_16_hyperparam_sweep_lr/n_images_13000704.pt
